# RAG Chatbot — Step by Step
### Ask questions from your PDF using Ollama + Pinecone

---
**What is RAG?**
- **R**etrieval → Find relevant parts from your PDF
- **A**ugmented → Add those parts to the question
- **G**eneration → LLM writes the answer

---
**Run the cells one by one from top to bottom ↓**

## Cell 1 — Install all required packages

In [1]:
# Install everything we need
# Run this cell only once

%pip install langchain langchain-community langchain-ollama pinecone-client langchain-pinecone pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Import all libraries

In [2]:
# Install missing package
# Upgrade all langchain packages together
%pip install --upgrade langchain langchain-core langchain-community langchain-ollama langchain-text-splitters pinecone-client langchain-pinecone pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Check exact version
import langchain
import langchain_community
import langchain_core
import langchain_ollama

print("langchain version        :", langchain.__version__)
print("langchain_community      :", langchain_community.__version__)
print("langchain_core           :", langchain_core.__version__)
print("langchain_ollama         :", langchain_ollama.__version__)

langchain version        : 1.2.17
langchain_community      : 0.4.1
langchain_core           : 1.3.2
langchain_ollama         : 1.1.0


In [4]:
# Import everything we need

from langchain_community.document_loaders import PyPDFLoader               # reads PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter         # splits into chunks
from langchain_ollama import OllamaEmbeddings                              # converts text to numbers
from langchain_ollama import OllamaLLM                                     # the AI that answers
from langchain_pinecone import PineconeVectorStore                         # saves and searches chunks
from langchain_core.prompts import ChatPromptTemplate                      # instruction for the AI
from pinecone import Pinecone, ServerlessSpec                              # Pinecone client

print("All libraries imported successfully!")

d:\Rag_Pinecone\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


## Cell 3 — Settings
> ⚠️ **Change these values to match your setup**

In [5]:
# ── Change these to match your setup ──────────────────────

PDF_FILE        = "text_medical.pdf"           # your PDF filename
EMBEDDING_MODEL = "embeddinggemma"                  # embedding model in Ollama
LLM_MODEL       = " tinyllama"                  # LLM model in Ollama
OLLAMA_URL      = "http://localhost:11434"   # Ollama always runs here

# ── Pinecone settings ──────────────────────────────────────
PINECONE_API_KEY   = "pcsk_2YNVM8_BsD6M67Kyg8wCeUR9yWrKjonnLj84Y4D9WBa8BcETZPYT5mfeq69ocXJwDtA5zR"   # get from https://app.pinecone.io
PINECONE_INDEX     = "rag"                # name of your Pinecone index
PINECONE_CLOUD     = "aws"                          # cloud provider: aws / gcp / azure
PINECONE_REGION    = "us-east-1"                    # region for your index
EMBEDDING_DIM      = 768                            # must match your embedding model output

# ──────────────────────────────────────────────────────────

print("Settings saved!")
print(f"  PDF File        : {PDF_FILE}")
print(f"  Embedding Model : {EMBEDDING_MODEL}")
print(f"  LLM Model       : {LLM_MODEL}")
print(f"  Ollama URL      : {OLLAMA_URL}")
print(f"  Pinecone Index  : {PINECONE_INDEX}")

Settings saved!
  PDF File        : text_medical.pdf
  Embedding Model : embeddinggemma
  LLM Model       :  tinyllama
  Ollama URL      : http://localhost:11434
  Pinecone Index  : rag


---
# Part A — Feed your PDF (run once)
---

## Cell 4 — Load the PDF

In [1]:
# Load your PDF file
# PyPDFLoader reads every page and gives us a list of documents

loader    = PyPDFLoader(PDF_FILE)
documents = loader.load()

print(f"PDF loaded successfully!")
print(f"Total pages found : {len(documents)}")
print(f"\nSample text from page 1:")
print("-" * 50)
print(documents[0].page_content[:400])
print("-" * 50)

NameError: name 'PyPDFLoader' is not defined

## Cell 5 — Split PDF into small chunks
> **Why?** The AI cannot read 100 pages at once. We split the PDF into small pieces so we can find the right piece for each question.

In [7]:
# Split the PDF into small chunks
# chunk_size    = how many characters in each chunk
# chunk_overlap = how many characters shared between chunks (avoids losing context)

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 200,
)

chunks = splitter.split_documents(documents)

print(f"Splitting done!")
print(f"Total chunks created : {len(chunks)}")
print(f"\nExample — Chunk 1 text:")
print("-" * 50)
print(chunks[0].page_content)
print("-" * 50)

Splitting done!
Total chunks created : 257

Example — Chunk 1 text:
--------------------------------------------------
A village health worker is a person who helps guide family members and neighbors toward better 
health. This person is often chosen by the community because he or she is kind, capable, and 
trustworthy. Some village health workers receive formal training and support from government 
programs such as the Ministry of Health. Others do not have any official title but are respected 
members of the community who help people with health problems. Many of them learn by 
observing others, assisting experienced workers, and studying on their own. 
In a broader sense, a village health worker is anyone who contributes to making the village a 
healthier place. Parents can teach their children the importance of cleanliness. Farmers can 
cooperate to grow enough nutritious food. Teachers can educate students about preventing and 
treating common illnesses and injuries. Children can 

## Cell 6 — Set up the Embedding Model
> **Why embeddings?** We convert text into numbers. Similar text gets similar numbers. This lets us search for relevant chunks using math.

In [8]:
# Connect to the Gemma embedding model running in Ollama

embedding_model = OllamaEmbeddings(
    model    = EMBEDDING_MODEL,
    base_url = OLLAMA_URL,
)

# Quick test — embed one sentence and check it works
test_vector = embedding_model.embed_query("This is a test sentence.")

print(f"Embedding model is working!")
print(f"Model used        : {EMBEDDING_MODEL}")
print(f"Each text becomes : {len(test_vector)} numbers")
print(f"First 5 numbers   : {test_vector[:5]}")

Embedding model is working!
Model used        : embeddinggemma
Each text becomes : 768 numbers
First 5 numbers   : [-0.14558502, 0.010534272, 0.030205863, -0.023637392, -0.03916916]


## Cell 7 — Save chunks into Pinecone
> **Why Pinecone?** It is a managed cloud vector database. It stores your embeddings online so you can search them fast from anywhere — no local folder needed.

In [9]:
# Convert all chunks to numbers and save in Pinecone
# This may take a few minutes depending on how big your PDF is

import os
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY

# Create Pinecone client and index (if it doesn't exist yet)
pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX not in pc.list_indexes().names():
    print(f'Creating new Pinecone index: {PINECONE_INDEX} ...')
    pc.create_index(
        name      = PINECONE_INDEX,
        dimension = EMBEDDING_DIM,
        metric    = 'cosine',
        spec      = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
    )
else:
    print(f'Index "{PINECONE_INDEX}" already exists, using it.')

print(f"Saving {len(chunks)} chunks to Pinecone...")
print("Please wait...")

vectorstore = PineconeVectorStore.from_documents(
    documents  = chunks,           # our text chunks
    embedding  = embedding_model,  # converts text to numbers
    index_name = PINECONE_INDEX,   # your Pinecone index
)

print(f"\nAll done!")
print(f"Chunks saved to Pinecone index : {PINECONE_INDEX}")

Index "rag" already exists, using it.
Saving 257 chunks to Pinecone...
Please wait...

All done!
Chunks saved to Pinecone index : rag


---
# Part B — Chat with your PDF
---

## Cell 8 — Load saved Pinecone index
> Next time you open this notebook, start from here. No need to re-run Part A.

In [10]:
# Load the Pinecone index we saved earlier
# No need to re-read the PDF again!

import os
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY

embedding_model = OllamaEmbeddings(
    model    = EMBEDDING_MODEL,
    base_url = OLLAMA_URL,
)

vectorstore = PineconeVectorStore(
    index_name        = PINECONE_INDEX,
    embedding         = embedding_model,
)

print(f"Pinecone index loaded!")
print(f"Index name : {PINECONE_INDEX}")

Pinecone index loaded!
Index name : rag


## Cell 9 — Set up the Retriever
> The retriever searches the database and finds the TOP 5 chunks most related to your question.

In [11]:
# Set up the retriever
# k=5 means it will find the 5 most relevant chunks

retriever = vectorstore.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 5}
)

print("Retriever is ready!")
print("It will find the top 5 most relevant chunks for each question.")

Retriever is ready!
It will find the top 5 most relevant chunks for each question.


## Cell 10 — Load the LLM
> The LLM reads the relevant chunks and writes a proper answer in plain English.

In [12]:
# Load the LLM from Ollama
# temperature = 0.1 means answers will be factual, not creative

llm = OllamaLLM(
    model       = LLM_MODEL,
    base_url    = OLLAMA_URL,
    temperature = 0.1,
)

print(f"LLM loaded successfully!")
print(f"Model : {LLM_MODEL}")

LLM loaded successfully!
Model :  tinyllama


## Cell 11 — Write the Prompt Template
> This is the instruction we give the AI. It tells the AI to only use information from the PDF and not make things up.

In [13]:
# This is the prompt template + llm connected with pipe |
# No extra imports needed — works perfectly on your version

rag_chain = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Use only the context below to answer the question.
If the answer is not in the context, say: "I don't know based on this document."
Do not make up any information.

Context from the document:
{context}

Question:
{input}

Answer:
""") | llm

print("RAG chain is ready!")

RAG chain is ready!


## Cell 12 — Build the RAG Chain
> This connects the retriever + LLM + prompt together into one pipeline.

In [14]:
# ── Change this question ──
question = "What is the best food for babies in the first six months?"
# ─────────────────────────

relevant_chunks = retriever.invoke(question)
context_text    = "\n\n".join(doc.page_content for doc in relevant_chunks)
answer          = rag_chain.invoke({"context": context_text, "input": question})

print(f"Question : {question}")
print("-" * 60)
print(f"Answer   : {answer}")
print("-" * 60)
print(f"Pages used : {set(str(doc.metadata.get('page','?')) for doc in relevant_chunks)}")

Question : What is the best food for babies in the first six months?
------------------------------------------------------------
Answer   : The best food for baie (infants) in the first six months of life is breast milk, which provides complete nutrition and protects against diahrreia and infectious diseases. Mother's milk remains nutritious even if she is thin or weak.
------------------------------------------------------------
Pages used : {'39.0', '41.0'}
